In [52]:
CREATE_COVER_LETTER_PROMPT = """
Write a professional, natural cover letter for the target job.

Use the EXISTING RESUME as the source of truth for the candidate.

IMPORTANT:
- Never invent or infer candidate experience.
- Never treat job requirements as candidate skills.
- Never claim the candidate has experience with something only because
  it appears in the job description.
- Preserve where information comes from in the resume.

The resume has different sections. Respect them:

EXPERIENCE = actual professional work experience.
PROJECTS = project experience.
TECHNICAL SKILLS = technologies/tools the candidate knows.
Do not turn a Technical Skill into professional experience unless the
Experience or Projects section explicitly supports it.

For example:
- TypeScript can be mentioned as a skill, and its use at Google can
  be mentioned as professional experience.
- React.js can be mentioned as a skill and its use in Code Fusion can
  be mentioned as project experience.
- Next.js can be mentioned as a skill and its use in JobPilot can be
  mentioned as project experience.
- AWS must NOT be mentioned because it is not in the resume.
- REST APIs, OAuth2, JWT, Jest, Playwright, Cypress, GitHub Actions,
  CI/CD, and web security must NOT be claimed because they are not
  explicitly supported by the resume.
- Do not claim 2+ years of professional experience. The resume shows
  a Google internship from May 2026 to July 2026.

Use JOB DATA only to identify which existing candidate experience,
projects, and skills are relevant to the role.

Do not copy the job description.

Do not invent reasons for wanting the company.
Do not mention company facts such as customer numbers unless the
candidate has a genuine, explicitly stated reason to mention them.

Write 250–350 words in 3–4 paragraphs.

Focus on:
- Google experience with Java and TypeScript.
- Relevant frontend/interface work at Google.
- Code Fusion and its React.js, MongoDB, Yjs, Socket.IO experience.
- JobPilot and its FastAPI, Next.js, Socket.IO, and AI/LLM experience.
- Other resume skills that genuinely match the job.

Do not repeat the same information.

Avoid generic phrases such as:
"ideal candidate", "perfect fit", "proven track record",
"passionate about", "thrilled to apply", "strong foundation",
"I am impressed by", or "make a significant contribution".

Return ONLY the finished cover letter.

USER DATA:
{user_object}

EXISTING RESUME:
{existing_resume}

JOB DATA:
{job_object}

ADDITIONAL USER INSTRUCTION:
{user_instruction}

{format_instructions}
"""

In [53]:
from pydantic import BaseModel, Field


class CreateCoverLetterResponse(BaseModel):
    cover_letter: str = Field(
        description=(
            "ONLY the complete professional cover letter. "
            "It must be ready to copy and send. "
            "Do not include explanations, analysis, metadata, "
            "headings such as 'Cover Letter', word counts, "
            "or any text outside the actual cover letter."
        )
    )

In [54]:
import os
import json

from dotenv import load_dotenv
from pydantic import ValidationError

from langchain_huggingface import (
    HuggingFaceEndpoint,
    ChatHuggingFace,
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.exceptions import OutputParserException

# from .create_cover_letter_prompt import CREATE_COVER_LETTER_PROMPT
# from .create_cover_letter_schema import CreateCoverLetterResponse


load_dotenv()


# =========================================================
# Model
# =========================================================

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    max_new_tokens=900,
    temperature=0.1,
)

model = ChatHuggingFace(llm=llm)


# =========================================================
# Pydantic Output Parser
# =========================================================

parser = PydanticOutputParser(
    pydantic_object=CreateCoverLetterResponse
)


# =========================================================
# Output Fixing Parser
# =========================================================

fixing_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=model,
    max_retries=2,
)


# =========================================================
# Prompt
# =========================================================

prompt = ChatPromptTemplate.from_template(
    CREATE_COVER_LETTER_PROMPT
)


# =========================================================
# Chain
# =========================================================

create_cover_letter_chain = (
    prompt
    | model
    | fixing_parser
)


# =========================================================
# Function
# =========================================================

def create_cover_letter(
    user: dict,
    resume: str,
    job: dict,
    user_instruction: str = "",
) -> CreateCoverLetterResponse:

    if not user:
        raise ValueError("user must not be empty")

    if not resume or not resume.strip():
        raise ValueError("resume must not be empty")

    if not job:
        raise ValueError("job must not be empty")

    # -----------------------------------------------------
    # Keep only useful candidate information
    # -----------------------------------------------------

    safe_user = {
        "full_name": user.get("full_name", ""),
        "email": user.get("email", ""),
        "phone": user.get("phone", ""),
        "linkedin_url": user.get("linkedin_url", ""),
        "github_url": user.get("github_url", ""),
        "portfolio_url": user.get("portfolio_url", ""),
    }
    
    safe_job = {
        "title": job.get("title", ""),
        "company": job.get("company", ""),
        "cities": job.get("cities", []),
        "countries": job.get("countries", []),
        "is_remote": job.get("is_remote", False),
        "is_hybride": job.get("is_hybride", False),
        "is_onsite": job.get("is_onsite", False),
        "required_skills": job.get("required_skills", []),
        "description": job.get("description", ""),
    }

    # -----------------------------------------------------
    # Generate cover letter
    # -----------------------------------------------------

    try:

        result = create_cover_letter_chain.invoke(
            {
                "user_object": json.dumps(
                    safe_user,
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                ),

                "existing_resume": resume.strip(),

                "job_object": json.dumps(
                    safe_job,
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                ),

                "user_instruction": (
                    user_instruction.strip()
                    if user_instruction
                    else "No additional instruction."
                ),

                "format_instructions": (
                    parser.get_format_instructions()
                ),
            }
        )

        return result

    except OutputParserException as e:

        print("Cover letter output parser failed:")
        print(e)

        raise

    except ValidationError as e:

        print("Cover letter Pydantic validation failed:")
        print(e)

        raise

    except Exception as e:

        print("Failed to generate cover letter:")
        print(e)

        raise

In [55]:
user = {
    "full_name": "Anirban Das",
    "email": "dasaniran268@gmail.com",
    "phone": "+91 629035587",
    "linkedin_url": "https://linkedin.com/in/anirban-das",
    "github_url": "https://github.com/anirban-das",
    "portfolio_url": "https://anirbandas.dev",
}

In [56]:
resume = """
# **Anirban Das** 

(+91) 629035587 _|_ dasaniran268@gmail.com _|_ LinkedIn _|_ GitHub 

## **Education** 

### **Indian Institute of Technology (ISM), Dhanbad, India** 

Bachelor of Technology (CGPA: 8.29/10) 2023 – **Relevant Coursework:** Data Structures & Algorithms, Object Oriented Programming (C++), Operating Systems, Database Management Systems (DBMS), Compiler Design 

2023 – 2027 

## **Experience** 

### **Google** _|_ **Software Engineer Intern** 

   - May 2026 – Jul 2026 

- Engineered a new linting rule and automated batch validation pipeline in Java and TypeScript to detect structural errors across cloud contract templates. 

- Enhanced core template management interfaces by developing a responsive document comparison dialog box, a read/write mode selector for gDoc, and an embedded AI assistant chat widget. 

- Contributed 7,350+ lines of production code across 22 peer-reviewed changelists and created analytical dashboards for contract monitoring and template health tracking. 

## **Projects** 

**Code Fusion** _|_ **React.js, MongoDB, Yjs, Socket.IO, ExpressJS** 

Deployed _|_ GitHub 

- Built a CRDT-based real-time collaborative code editor using Yjs and Monaco Editor, enabling conflict-free multi-user editing through delta-based synchronization and awareness protocol for live cursor tracking across sessions. 

- Developed a real-time collaboration system supporting live cursor tracking and integrated in-app chat, allowing seamless communication between distributed developers during shared coding sessions. 

- Implemented multi-language support, auto-completion, real-time syntax error detection, persistent code storage, customizable themes, and a collapsible file explorer for efficient workflow management. 

### **JobPilot** _|_ **FastAPI, Next.js, Socket.IO, LangChain, Tailwind CSS** 

   - GitHub 

- Developed an automation system using FastAPI, Next.js, and HuggingFace LLMs to automate resume extraction and job matching, reducing manual application time by 80%. 

- Architected a scalable asynchronous worker system with WebSockets/Socket.IO updates and FileLock synchronization to manage concurrent multi-user job lifecycles. 

- Built an LLM-powered job ranking and categorization pipeline using prompt engineering, along with a manual review workflow and local validation setup. 

## **Competitive Programming** 

**LeetCode:** aswU2SZvDg _|_ Solved 240+ problems focusing on data structures and algorithms **CodeForces:** anirban2005 _|_ Max Rating: 1216 (Pupil) _|_ Solved 390+ problems 

## **Technical Skills** 

- **Languages & Databases:** C++, Python, JavaScript, TypeScript, MongoDB, PostgreSQL, Vector Databases 

- **Core Frameworks:** React.js, Next.js, Node.js, Express.js, FastAPI, Tailwind CSS, Three.js 

- **AI/ML & Tools:** LangChain, LangGraph, TensorFlow, Keras, Git, Docker, Postman 

## **Achievements** 

- Secured **4th** rank at **HaXplore** | **CodeFest’25** , organized by **IIT BHU** . 

- **Winner** of Winter of Code 6.0 (Web Development Division), a one-month long hackathon conducted by **CyberLabs** , IIT (ISM) Dhanbad. | **Deployed Project** 

"""

In [57]:
job = {
    "_id": "6a835b392645704f6b223099",
    "sourceId": "6a835b392645704f6b223098",
    "jobId": "job-168",

    "cities": [
        "Hyderabad",
        "Pune"
    ],

    "company": "FinStack Labs",

    "countries": [
        "India"
    ],

    "createdAt": "2026-08-18T05:30:00.000Z",

    "description": """
We are looking for a Full Stack Software Engineer to build and maintain
high-quality financial technology products used by thousands of customers.

The engineer will work across frontend and backend systems and collaborate
closely with product managers, designers, QA engineers, and infrastructure
teams.

Responsibilities:

- Design and develop responsive web applications using React and Next.js.
- Build backend APIs and services using Node.js and TypeScript.
- Develop reusable frontend components and maintain a scalable frontend
  architecture.
- Design and optimize MongoDB data models and database queries.
- Integrate third-party APIs and external services.
- Implement authentication, authorization, and secure API endpoints.
- Write unit, integration, and end-to-end tests.
- Containerize applications using Docker and support CI/CD workflows.
- Monitor application performance and troubleshoot production issues.
- Participate in code reviews and contribute to engineering standards.
- Work with product and design teams to deliver customer-facing features.

Requirements:

- 2+ years of professional software engineering experience.
- Strong proficiency in TypeScript and JavaScript.
- Strong experience with React and Next.js.
- Experience building backend services with Node.js.
- Solid understanding of MongoDB and database design.
- Experience developing and consuming REST APIs.
- Experience with Docker and containerized applications.
- Familiarity with AWS services such as ECS, Lambda, S3, and CloudWatch.
- Experience with automated testing using Jest, Playwright, or Cypress.
- Understanding of authentication mechanisms such as OAuth2 and JWT.
- Familiarity with CI/CD pipelines and GitHub Actions.
- Strong understanding of web application security and performance
  optimization.

Nice to have:

- Experience with PostgreSQL.
- Experience with Redis.
- Experience working with WebSockets.
- Experience with real-time collaborative applications.
- Experience working in a fast-paced startup environment.
- Experience with AI-powered product features.
    """,

    "is_hybride": True,
    "is_onsite": False,
    "is_remote": True,

    "required_skills": [
        "TypeScript",
        "JavaScript",
        "React",
        "Next.js",
        "Node.js",
        "MongoDB",
        "REST APIs",
        "Docker",
        "AWS",
        "ECS",
        "Lambda",
        "S3",
        "CloudWatch",
        "Jest",
        "Playwright",
        "Cypress",
        "OAuth2",
        "JWT",
        "GitHub Actions",
        "CI/CD",
        "Web Security",
        "Performance Optimization"
    ],

    "salary_offered": "₹24,00,000 - ₹36,00,000 per year",

    "start_date": "2026-11-01",

    "status": "discovered",

    "title": "Full Stack Software Engineer",

    "updatedAt": "2026-08-18T05:30:00.000Z",

    "visa_sponsorship_offered": False
}

In [58]:
cover_letter = create_cover_letter(
    user=user,
    resume=resume,
    job=job,
)

print(cover_letter.cover_letter)

Dear Hiring Manager at FinStack Labs,

I am excited to apply for the Full Stack Software Engineer position at FinStack Labs. With a strong foundation in software engineering, I am confident that my skills and experience make me an ideal candidate for this role.

As a software engineer intern at Google, I gained hands-on experience with Java and TypeScript, developing a new linting rule and automated batch validation pipeline to detect structural errors across cloud contract templates. I also enhanced core template management interfaces by creating a responsive document comparison dialog box, a read/write mode selector for gDoc, and an embedded AI assistant chat widget. My contributions included 7,350+ lines of production code across 22 peer-reviewed changelists and analytical dashboards for contract monitoring and template health tracking.

In my project experience at Code Fusion, I built a CRDT-based real-time collaborative code editor using Yjs and Monaco Editor, enabling conflict-fr